### PulmoTriage · Clinical-Note Model (NLP)

###  Setup

#### Install

In [ ]:
!pip install -q optuna mlflow captum

#### Input files

In [ ]:
import os, glob

def find_file(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return hits[0] if hits else None

DATA_CSV     = find_file('nlp_dataset.csv')
CLASSES_JSON = find_file('mlb_classes.json')
REPORTS_CSV  = find_file('indiana_reports.csv')
OUT_DIR      = '/kaggle/working'
assert DATA_CSV and CLASSES_JSON, 'Add the preprocessing output (nlp_dataset.csv, mlb_classes.json) as an input'
print(DATA_CSV, CLASSES_JSON, REPORTS_CSV, sep='\n')

#### Settings

In [ ]:
SEED = 42
FAST = False

CANDIDATE_MODELS = {
    'Bio_ClinicalBERT': 'emilyalsentzer/Bio_ClinicalBERT',
    'BiomedBERT':       'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
    'DistilBERT':       'distilbert-base-uncased',
}

In [ ]:
CV_FOLDS   = 2 if FAST else 3
N_TRIALS   = 4 if FAST else 12
MAX_EPOCHS = 4 if FAST else 10
PATIENCE   = 2 if FAST else 3
N_ENSEMBLE = 2 if FAST else 5
MIN_POS    = 5

#### Imports

In [ ]:
import re, gc, json, time, shutil, random, warnings, zipfile
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoConfig, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, hamming_loss, accuracy_score
import optuna, mlflow, joblib
from IPython.display import HTML

warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)

#### Reproducibility, device and MLflow

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('Device:', device)

mlflow.set_tracking_uri(f'sqlite:///{OUT_DIR}/mlflow.db')
mlflow.set_experiment('text_model')
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)

### Text module (`pulmo_text.py`)

In [ ]:
%%writefile pulmo_text.py
import re, json, os, glob, string
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

_MISSING = {'none', 'nan', 'none.', 'none available', 'n/a', 'na', 'null', '', 'xxxx', 'xxxx, xxxx.', 'xxxx, xxxx', 'xxxx.'}
_PUNCT = str.maketrans('', '', string.punctuation)
_NON_INFO = re.compile(r"\b(hiddenyearold|hiddenyearsold|yearold|yearsold|male|female|patient)\b")

def clean_indication(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    t = re.sub(r"\s+", " ", str(text).strip().lower())
    t = t.translate(_PUNCT)
    if t in _MISSING or t == "not provided":
        return ""
    t = re.sub(r"(?i)xxxx", "hidden", t)
    rest = re.sub(r"\s+", " ", _NON_INFO.sub("", t)).strip().replace("hidden", "").strip()
    if len(rest) == 0 or rest in ["with", "old", "year", "and", "or"]:
        return ""
    return t

class TextModel:

    def __init__(self, model_dir, device=None):
        with open(os.path.join(model_dir, "meta.json")) as f:
            self.meta = json.load(f)
        self.dir = model_dir
        self.classes = list(self.meta["classes"])
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.models = [AutoModelForSequenceClassification.from_pretrained(d).float().to(self.device).eval()
                       for d in sorted(glob.glob(os.path.join(model_dir, "fold_*")))]
        self.model = self.models[0]
        self.tfidf = self._load("tfidf_ovr.joblib")
        self.emb_lr = self._load("emb_lr.joblib")
        self.w = self.meta["blend_weights"]
        self.max_len = int(self.meta["max_len"])
        self.thresholds = np.asarray(self.meta["thresholds"], dtype=np.float32)
        self._encoder = None

    def _load(self, name):
        path = os.path.join(self.dir, name)
        if os.path.exists(path):
            import joblib
            return joblib.load(path)
        return None

    def _enc(self, texts):
        enc = self.tok(list(texts), truncation=True, max_length=self.max_len, padding=True, return_tensors="pt")
        return {k: v.to(self.device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}

    @torch.no_grad()
    def _finetuned(self, clean, batch_size=64):
        out = np.zeros((len(clean), len(self.classes)), dtype=np.float32)
        for s in range(0, len(clean), batch_size):
            enc = self._enc(clean[s:s + batch_size])
            out[s:s + batch_size] = torch.stack([torch.sigmoid(m(**enc).logits.float()) for m in self.models]).mean(0).cpu().numpy()
        return out

    def predict_proba(self, texts, batch_size=64):
        clean = [clean_indication(t) for t in texts]
        idx = [i for i, c in enumerate(clean) if c]
        p = np.full((len(clean), len(self.classes)), np.nan, dtype=np.float32)
        if idx:
            sub = [clean[i] for i in idx]
            mix = self.w["finetuned"] * self._finetuned(sub, batch_size)
            if self.w["tfidf"] > 0:
                mix += self.w["tfidf"] * self.tfidf.predict_proba(sub)
            if self.w["embeddings"] > 0:
                mix += self.w["embeddings"] * self.emb_lr.predict_proba(self.embed(sub))
            p[idx] = mix
        return p

    def predict_text(self, text, top_k=5):
        clean = clean_indication(text)
        if not clean:
            return {"has_note": False, "findings": [], "top": [], "warning": "No usable clinical note"}
        p = self.predict_proba([text])[0]
        order = np.argsort(-p)
        item = lambda i: {"label": self.classes[i], "probability": round(float(p[i]), 4)}
        out = {"has_note": True, "clean_text": clean,
               "findings": [item(i) for i in order if p[i] > self.thresholds[i]],
               "top": [item(i) for i in order[:top_k]]}
        if "normal" in self.classes:
            out["p_normal"] = round(float(p[self.classes.index("normal")]), 4)
        return out

    @torch.no_grad()
    def embed(self, texts, batch_size=64):
        if self._encoder is None:
            self._encoder = AutoModel.from_pretrained(os.path.join(self.dir, "frozen_encoder")).float().to(self.device).eval()
        clean = [clean_indication(t) for t in texts]
        dim = getattr(self._encoder.config, "hidden_size", None) or self._encoder.config.dim
        out = np.zeros((len(clean), dim), dtype=np.float32)
        idx = [i for i, c in enumerate(clean) if c]
        for s in range(0, len(idx), batch_size):
            b = idx[s:s + batch_size]
            enc = self._enc([clean[i] for i in b])
            h = self._encoder(**enc).last_hidden_state.float()
            m = enc["attention_mask"].unsqueeze(-1).float()
            out[b] = ((h * m).sum(1) / m.sum(1)).cpu().numpy()
        return out

    def explain(self, text, label=None, top_k=8, n_steps=50):
        from captum.attr import LayerIntegratedGradients
        clean = clean_indication(text)
        if not clean:
            return {"label": None, "tokens": [], "top": []}
        if label is None:
            p = self.predict_proba([text])[0]
            label = next((self.classes[i] for i in np.argsort(-p) if self.classes[i] != "normal"), self.classes[int(np.argmax(p))])
        k = self.classes.index(label)
        enc = self._enc([clean])
        ids, mask = enc["input_ids"], enc["attention_mask"]
        special = {i for i in (self.tok.cls_token_id, self.tok.sep_token_id, self.tok.pad_token_id,
                               self.tok.bos_token_id, self.tok.eos_token_id) if i is not None}
        base = ids.clone()
        base[0, [j for j in range(ids.shape[1]) if int(ids[0, j]) not in special]] = self.tok.pad_token_id
        fwd = lambda input_ids, attention_mask: torch.sigmoid(self.model(input_ids=input_ids, attention_mask=attention_mask).logits.float())[:, k]
        attr = LayerIntegratedGradients(fwd, self.model.get_input_embeddings()).attribute(
            ids, baselines=base, additional_forward_args=(mask,), n_steps=n_steps, internal_batch_size=25)
        scores = attr.sum(-1).squeeze(0).detach().cpu().numpy()
        words, wscores = [], []
        for tkn, sc, tid in zip(self.tok.convert_ids_to_tokens(ids[0].tolist()), scores, ids[0].tolist()):
            if tid in special:
                continue
            if tkn.startswith("##") and words:
                words[-1] += tkn[2:]; wscores[-1] += sc
            else:
                words.append(tkn); wscores.append(sc)
        norm = float(np.abs(wscores).max()) or 1.0 if wscores else 1.0
        items = [{"word": w, "score": round(float(s / norm), 4)} for w, s in zip(words, wscores)]
        return {"label": label, "tokens": items, "top": sorted(items, key=lambda d: -abs(d["score"]))[:top_k]}

In [ ]:
import importlib, pulmo_text
importlib.reload(pulmo_text)
from pulmo_text import clean_indication
for s in ['XXXX-year-old male with chest pain', 'SOB. Hx of COPD.', 'XXXX XXXX', None]:
    print(repr(s), '->', repr(clean_indication(s)))

### Data

Cleaned notes, the train / validation / test split and the multi-label targets all come from the preprocessing notebook.

#### Load

In [ ]:
data = pd.read_csv(DATA_CSV, keep_default_na=False, na_values=[''])
CLASSES = json.load(open(CLASSES_JSON))

df = data.drop_duplicates('uid').reset_index(drop=True)
df['split'] = df['split'].astype(str).str.lower()
df['text'] = df['indication'].fillna('not provided').astype(str).str.strip()
df['has_note'] = ~df['text'].isin(['not provided', ''])
df['findings_text'] = df['findings'].fillna('').astype(str).replace('not provided', '')
df['strat'] = df['is_abnormal'].astype(str)
df['y'] = list(df[CLASSES].values.astype(np.float32))
print(f'{len(df)} patients | {len(CLASSES)} findings')
df[['uid', 'split', 'text']].head()

#### Check: the app's cleaning gives the same text as the preprocessing notebook

In [ ]:
if REPORTS_CSV:
    orig = pd.read_csv(REPORTS_CSV)[['uid', 'indication']].merge(df[['uid', 'text']], on='uid')
    same = (orig['indication'].apply(clean_indication).replace('', 'not provided') == orig['text']).mean()
    print(f'Same cleaned text for {same:.1%} of notes')

#### Split sizes

In [ ]:
df.groupby('split').agg(patients=('uid', 'size'), with_note=('has_note', 'sum')).reindex(['train', 'val', 'test'])

#### How many cases each finding has

In [ ]:
counts = df[df.split == 'train'][CLASSES].sum().sort_values(ascending=False)
print(f'Findings with >= {MIN_POS} training cases: {(counts >= MIN_POS).sum()} of {len(CLASSES)}')
counts.head(25).iloc[::-1].plot.barh(figsize=(8, 6), color='#1428A0', title='Most frequent findings (training set)')
plt.xlabel('cases'); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/finding_counts.png', dpi=120); plt.show()

#### Notes used for training and evaluation (patients without a note are skipped)

In [ ]:
fit_df  = df[(df.split == 'train') & df.has_note].reset_index(drop=True)
val_df  = df[(df.split == 'val')   & df.has_note].reset_index(drop=True)
test_df = df[(df.split == 'test')  & df.has_note].reset_index(drop=True)
Ymat = lambda frame: np.stack(frame['y'].values)
print(f'train {len(fit_df)} | val {len(val_df)} | test {len(test_df)}')

#### Note length in tokens -> MAX_LEN

In [ ]:
first_tok = AutoTokenizer.from_pretrained(list(CANDIDATE_MODELS.values())[0])
lens = np.array([len(first_tok(t)['input_ids']) for t in fit_df.text])
MAX_LEN = int(min(128, 8 * np.ceil(np.percentile(lens, 99.5) / 8)))
print(f'Tokens per note: median {np.median(lens):.0f}, max {lens.max()} -> MAX_LEN = {MAX_LEN}')

### Metrics

#### Scores

In [ ]:
def scored(Y):
    pos = Y.sum(0)
    return (pos >= MIN_POS) & (pos < len(Y))

def main_score(Y, P):
    Y, P = np.asarray(Y), np.asarray(P)
    ok = scored(Y)
    return roc_auc_score(Y[:, ok], P[:, ok], average='macro') if ok.any() else roc_auc_score(Y.ravel(), P.ravel())

In [ ]:
def foldwise_score(Y, P, folds):
    return float(np.mean([main_score(Y[b], P[b]) for _, b in folds]))

def all_metrics(Y, P, thr):
    Y, P = np.asarray(Y), np.asarray(P); pred = (P > thr).astype(int); ok = scored(Y)
    return {'macro_roc_auc': main_score(Y, P), 'micro_roc_auc': roc_auc_score(Y.ravel(), P.ravel()),
            'micro_pr_auc': average_precision_score(Y.ravel(), P.ravel()),
            'macro_pr_auc': average_precision_score(Y[:, ok], P[:, ok], average='macro'),
            'micro_f1': f1_score(Y, pred, average='micro', zero_division=0),
            'macro_f1': f1_score(Y[:, ok], pred[:, ok], average='macro', zero_division=0),
            'micro_precision': precision_score(Y, pred, average='micro', zero_division=0),
            'micro_recall': recall_score(Y, pred, average='micro', zero_division=0),
            'hamming_loss': hamming_loss(Y, pred), 'exact_match': accuracy_score(Y, pred),
            'scored_findings': int(ok.sum())}

#### Decision thresholds

In [ ]:
def best_cut(y, p, f1_fn):
    cands = np.unique(np.quantile(p, np.linspace(0.5, 0.999, 80)))
    ok = [t for t in cands if (p > t).mean() <= 2 * y.mean()]
    f1s = [f1_fn(t) for t in ok]
    if not ok or max(f1s) <= 0:
        return 1.01
    return float(ok[int(np.argmax(f1s))])

In [ ]:
def global_threshold(Y, P):
    return best_cut(Y.ravel(), P.ravel(), lambda t: f1_score(Y, (P > t).astype(int), average='micro', zero_division=0))

def finding_thresholds(Y, P):
    thr = np.full(Y.shape[1], global_threshold(Y, P), dtype=np.float32)
    for j in np.where(Y.sum(0) >= MIN_POS)[0]:
        thr[j] = best_cut(Y[:, j], P[:, j], lambda t: f1_score(Y[:, j], (P[:, j] > t).astype(int), zero_division=0))
    return thr

#### Bootstrap confidence interval

In [ ]:
def boot_ci(Y, P, n=300):
    rng = np.random.default_rng(SEED); vals = []
    for _ in range(n):
        i = rng.integers(0, len(Y), len(Y))
        try: vals.append(main_score(Y[i], P[i]))
        except ValueError: pass
    return np.percentile(vals, [2.5, 97.5])

#### Model 1 : TF-IDF + logistic regression

One logistic regression per finding on word n-grams (and optionally character n-grams, which catch abbreviations such as *sob*, *hx*, *copd*).

#### Model definition

In [ ]:
def make_tfidf(C=1.0, chars=False):
    feats = [('words', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True))]
    if chars:
        feats.append(('chars', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, sublinear_tf=True)))
    return make_pipeline(FeatureUnion(feats), OneVsRestClassifier(LogisticRegression(C=C, max_iter=2000)))

#### Choose the settings (cross-validation on training notes)

In [ ]:
cv = list(StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED).split(fit_df, fit_df['strat']))
Yfit, rows = Ymat(fit_df), []
for chars in (False, True):
    for C in ((1.0, 4.0) if FAST else (0.3, 1.0, 3.0, 10.0)):
        oof = np.zeros(Yfit.shape)
        for a, b in cv:
            oof[b] = make_tfidf(C, chars).fit(fit_df.text.iloc[a], Yfit[a]).predict_proba(fit_df.text.iloc[b])
        rows.append({'char_ngrams': chars, 'C': C, 'cv_macro_roc_auc': foldwise_score(Yfit, oof, cv)})
tfidf_grid = pd.DataFrame(rows).sort_values('cv_macro_roc_auc', ascending=False).reset_index(drop=True)
TFIDF_CFG = {'C': float(tfidf_grid.C[0]), 'chars': bool(tfidf_grid.char_ngrams[0])}
print('Chosen:', TFIDF_CFG)
tfidf_grid.round(4)

#### Reference score (trained on train, evaluated on test)

In [ ]:
ref = make_tfidf(**TFIDF_CFG).fit(fit_df.text, Ymat(fit_df))
P_test_ref = ref.predict_proba(test_df.text)
with mlflow.start_run(run_name='tfidf_reference'):
    mlflow.log_params(TFIDF_CFG); mlflow.log_metric('test_macro_roc_auc', main_score(Ymat(test_df), P_test_ref))
print(f'TF-IDF test macro ROC-AUC: {main_score(Ymat(test_df), P_test_ref):.3f}')

### Model 2 : Fine-tuned transformer

#### Data loader

In [ ]:
_tok_cache = {}
def get_tok(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name)
    return _tok_cache[name]

def make_loader(frame, tok, bs, shuffle, text_col='text'):
    data = list(zip(frame[text_col].tolist(), frame['y'].tolist()))
    def collate(batch):
        t, y = zip(*batch)
        enc = tok(list(t), truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'],
                'labels': torch.tensor(np.stack(y), dtype=torch.float32)}
    return DataLoader(data, batch_size=bs, shuffle=shuffle, collate_fn=collate, generator=torch.Generator().manual_seed(SEED))

#### Model (output layer starts at each finding's frequency)

In [ ]:
def build_model(name, dropout, prior):
    cfg = AutoConfig.from_pretrained(name, num_labels=len(CLASSES), problem_type='multi_label_classification')
    if hasattr(cfg, 'seq_classif_dropout'): cfg.seq_classif_dropout = dropout
    else: cfg.classifier_dropout = dropout
    model = AutoModelForSequenceClassification.from_pretrained(name, config=cfg, ignore_mismatched_sizes=True)
    p = np.clip(prior, 1e-4, 1 - 1e-4)
    with torch.no_grad():
        model.classifier.bias.copy_(torch.tensor(np.log(p / (1 - p)), dtype=torch.float32))
    return model

#### Optimizer with layer-wise learning rates

In [ ]:
def make_optimizer(model, hp, n_steps):
    n_layers = getattr(model.config, 'num_hidden_layers', None) or model.config.n_layers
    groups = {}
    for n, p in model.named_parameters():
        m = re.search(r'layer\.(\d+)\.', n)
        depth = int(m.group(1)) + 1 if m else (0 if 'embeddings' in n else n_layers + 1)
        lr = hp['lr'] * hp['head_lr_mult'] if depth == n_layers + 1 else hp['lr'] * hp['llrd'] ** (n_layers + 1 - depth)
        no_wd = n.endswith('bias') or 'LayerNorm' in n or 'layer_norm' in n
        groups.setdefault((depth, no_wd), {'params': [], 'lr': lr, 'weight_decay': 0.0 if no_wd else hp['weight_decay']})['params'].append(p)
    opt = torch.optim.AdamW(list(groups.values()))
    return opt, get_linear_schedule_with_warmup(opt, int(hp['warmup'] * n_steps), n_steps)

#### Loss (mild weights for rare findings + smoothing towards each finding's rate)

In [ ]:
def make_loss(Ytr, hp):
    pos = Ytr.sum(0)
    weight = torch.tensor(np.clip(np.sqrt((len(Ytr) - pos) / np.maximum(pos, 1)), 1, 3), dtype=torch.float32, device=device)
    prior = torch.tensor(Ytr.mean(0), dtype=torch.float32, device=device)
    ls = hp['label_smoothing']
    bce = nn.BCEWithLogitsLoss(pos_weight=weight)
    return lambda logits, y: bce(logits.float(), y * (1 - ls) + ls * prior)

#### Predict and train one epoch

In [ ]:
@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); out = []
    for b in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device)).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

In [ ]:
def train_one_epoch(model, loader, opt, sched, scaler, loss_fn):
    model.train(); total = 0.0
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=b['input_ids'], attention_mask=b['attention_mask']).logits
        loss = loss_fn(logits, b['labels'])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    return total / len(loader)

#### Full training (early stopping, or a fixed number of epochs for the final ensemble)

In [ ]:
def train_model(name, hp, tr_df, va_df, text_col='text', verbose=False, early_stop=True):
    set_seed()
    tok, Ytr, Yva = get_tok(name), Ymat(tr_df), Ymat(va_df)
    model = build_model(name, hp['dropout'], Ytr.mean(0)).to(device)
    tl, vl = make_loader(tr_df, tok, hp['batch_size'], True, text_col), make_loader(va_df, tok, 64, False, text_col)
    opt, sched = make_optimizer(model, hp, len(tl) * hp['epochs'])
    loss_fn, scaler = make_loss(Ytr, hp), torch.amp.GradScaler(device.type, enabled=USE_AMP)
    best, best_state, best_probs, bad, history = -1, None, None, 0, []
    for ep in range(hp['epochs']):
        loss = train_one_epoch(model, tl, opt, sched, scaler, loss_fn)
        probs = predict_probs(model, vl); score = main_score(Yva, probs)
        history.append({'epoch': ep + 1, 'loss': loss, 'val_macro_roc_auc': score})
        if verbose: print(f'  epoch {ep + 1}: loss {loss:.4f} | val macro ROC-AUC {score:.4f}')
        if not early_stop:
            best, best_probs = score, probs
        elif score > best + 1e-4:
            best, best_probs, bad = score, probs, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    if early_stop: model.load_state_dict(best_state)
    best_epoch = int(np.argmax([h['val_macro_roc_auc'] for h in history]) + 1)
    return model, tok, {'score': best, 'best_epoch': best_epoch, 'history': history}, best_probs

#### Cross-validation helper

In [ ]:
def free():
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

def cv_score(name, hp, trial=None):
    scores, epochs = [], []
    for k, (a, b) in enumerate(cv):
        model, _, info, _ = train_model(name, hp, fit_df.iloc[a], fit_df.iloc[b])
        scores.append(info['score']); epochs.append(info['best_epoch'])
        del model; free()
        if trial is not None:
            trial.report(float(np.mean(scores)), k)
            if trial.should_prune(): raise optuna.TrialPruned()
    return {'cv_macro_roc_auc': float(np.mean(scores)), 'cv_std': float(np.std(scores)), 'best_epoch': float(np.mean(epochs))}

### Choosing the pre-trained model

Same recipe, same folds; the best mean cross-validated score wins.

#### Compare the candidates

In [ ]:
DEFAULT_HP = {'lr': 3e-5, 'llrd': 0.9, 'weight_decay': 0.01, 'warmup': 0.1, 'batch_size': 16,
              'epochs': MAX_EPOCHS, 'dropout': 0.1, 'label_smoothing': 0.0, 'head_lr_mult': 20.0}
rows = []
for short, name in CANDIDATE_MODELS.items():
    with mlflow.start_run(run_name=f'backbone_{short}'):
        res = cv_score(name, DEFAULT_HP)
        mlflow.log_params({**DEFAULT_HP, 'backbone': name}); mlflow.log_metrics(res)
    rows.append({'model': short, **res})
    print(f"{short:17s} {res['cv_macro_roc_auc']:.4f} ± {res['cv_std']:.4f}")
backbones = pd.DataFrame(rows).sort_values('cv_macro_roc_auc', ascending=False).reset_index(drop=True)
BEST_SHORT = backbones.model[0]; BEST_NAME = CANDIDATE_MODELS[BEST_SHORT]
print('Chosen:', BEST_SHORT)
backbones.round(4)

#### Chart

In [ ]:
plt.figure(figsize=(6, 2.8))
plt.barh(backbones.model, backbones.cv_macro_roc_auc, xerr=backbones.cv_std, color='#1428A0')
plt.xlim(0.45, min(1, backbones.cv_macro_roc_auc.max() + 0.06)); plt.gca().invert_yaxis()
plt.xlabel('cross-validated macro ROC-AUC'); plt.title('Pre-trained model comparison')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/backbone_comparison.png', dpi=120); plt.show()

### Hyper-parameter tuning (Optuna)

Bayesian search scored by cross-validated macro ROC-AUC; weak trials are stopped early.

#### Search space

In [ ]:
def objective(trial):
    hp = {'lr':              trial.suggest_float('lr', 1e-5, 8e-5, log=True),
          'llrd':            trial.suggest_float('llrd', 0.7, 1.0),
          'weight_decay':    trial.suggest_float('weight_decay', 0.0, 0.1),
          'warmup':          trial.suggest_float('warmup', 0.0, 0.2),
          'batch_size':      trial.suggest_categorical('batch_size', [8, 16, 32]),
          'dropout':         trial.suggest_float('dropout', 0.0, 0.3),
          'label_smoothing': trial.suggest_float('label_smoothing', 0.0, 0.15),
          'head_lr_mult':    trial.suggest_float('head_lr_mult', 1.0, 100.0, log=True),
          'epochs':          MAX_EPOCHS}
    with mlflow.start_run(run_name=f'optuna_{trial.number}', nested=True):
        mlflow.log_params(hp)
        res = cv_score(BEST_NAME, hp, trial)
        mlflow.log_metrics(res)
    trial.set_user_attr('best_epoch', res['best_epoch'])
    return res['cv_macro_roc_auc']

#### Run the search (longest step)

In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
study.enqueue_trial({k: v for k, v in DEFAULT_HP.items() if k != 'epochs'})
with mlflow.start_run(run_name=f'optuna_{BEST_SHORT}'):
    study.optimize(objective, n_trials=N_TRIALS)
    mlflow.log_metric('best_cv_macro_roc_auc', study.best_value)
print(f'Best CV macro ROC-AUC {study.best_value:.4f} (default recipe {backbones.cv_macro_roc_auc[0]:.4f})')
study.trials_dataframe(attrs=('number', 'value', 'state', 'params')).sort_values('value', ascending=False).head(8).round(5)

#### Tuning plots

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
try:
    plot_optimization_history(study); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/optuna_history.png', dpi=120); plt.show()
    plot_param_importances(study); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/optuna_importance.png', dpi=120); plt.show()
except Exception as e:
    print('Plots skipped:', e)

### Final model

#### Settings for the final models

In [ ]:
FINAL_HP = {**DEFAULT_HP, **study.best_params}
FINAL_HP['epochs'] = int(max(2, round(study.best_trial.user_attrs['best_epoch'])))
dev_df = pd.concat([fit_df, val_df], ignore_index=True)
Ydev, Ytest = Ymat(dev_df), Ymat(test_df)
folds = list(StratifiedKFold(N_ENSEMBLE, shuffle=True, random_state=SEED).split(dev_df, dev_df['strat']))
MODEL_DIR = f'{OUT_DIR}/text_model'
shutil.rmtree(MODEL_DIR, ignore_errors=True); os.makedirs(MODEL_DIR)
print('Model:', BEST_SHORT)
pd.Series(FINAL_HP).to_frame('final setting')

#### Model 2: ensemble of fine-tuned models

In [ ]:
oof_ft = np.zeros(Ydev.shape, dtype=np.float32); test_ft, histories = [], []
for k, (a, b) in enumerate(folds):
    print(f'Model {k + 1}/{N_ENSEMBLE}')
    m, tok, info, oof_ft[b] = train_model(BEST_NAME, FINAL_HP, dev_df.iloc[a], dev_df.iloc[b], verbose=True, early_stop=False)
    test_ft.append(predict_probs(m, make_loader(test_df, tok, 64, False)))
    histories.append(info['history'])
    m.half().save_pretrained(f'{MODEL_DIR}/fold_{k}')
    del m; free()
tok.save_pretrained(MODEL_DIR)
P_test_ft = np.mean(test_ft, axis=0)
print(f'Out-of-fold macro ROC-AUC: {foldwise_score(Ydev, oof_ft, folds):.4f}')

#### Model 1: TF-IDF on the same folds

In [ ]:
oof_tf = np.zeros(Ydev.shape, dtype=np.float32)
for a, b in folds:
    oof_tf[b] = make_tfidf(**TFIDF_CFG).fit(dev_df.text.iloc[a], Ydev[a]).predict_proba(dev_df.text.iloc[b])
tfidf_final = make_tfidf(**TFIDF_CFG).fit(dev_df.text, Ydev)
P_test_tf = tfidf_final.predict_proba(test_df.text)
joblib.dump(tfidf_final, f'{MODEL_DIR}/tfidf_ovr.joblib')
print(f'Out-of-fold macro ROC-AUC: {foldwise_score(Ydev, oof_tf, folds):.4f}')

#### Model 3: frozen-encoder embeddings

In [ ]:
encoder = AutoModel.from_pretrained(BEST_NAME).to(device).eval()

@torch.no_grad()
def frozen_embed(texts, bs=64):
    tk, out = get_tok(BEST_NAME), []
    for s in range(0, len(texts), bs):
        e = tk(texts[s:s + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        e = {k: v.to(device) for k, v in e.items() if k in ('input_ids', 'attention_mask')}
        h, msk = encoder(**e).last_hidden_state.float(), e['attention_mask'].unsqueeze(-1).float()
        out.append(((h * msk).sum(1) / msk.sum(1)).cpu().numpy())
    return np.concatenate(out)

In [ ]:
E_dev, E_test = frozen_embed(dev_df.text.tolist()), frozen_embed(test_df.text.tolist())
encoder.half().save_pretrained(f'{MODEL_DIR}/frozen_encoder'); del encoder; free()
print('Embeddings:', E_dev.shape)

#### Model 3: logistic regression on the embeddings

In [ ]:
make_emb_lr = lambda C: make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(C=C, max_iter=3000)))
best_emb = (-1, None, None)
for C in ((0.05,) if FAST else (0.01, 0.05, 0.2)):
    o = np.zeros(Ydev.shape, dtype=np.float32)
    for a, b in folds:
        o[b] = make_emb_lr(C).fit(E_dev[a], Ydev[a]).predict_proba(E_dev[b])
    best_emb = max(best_emb, (foldwise_score(Ydev, o, folds), C, o), key=lambda r: r[0])
_, EMB_C, oof_emb = best_emb
emb_final = make_emb_lr(EMB_C).fit(E_dev, Ydev)
P_test_emb = emb_final.predict_proba(E_test)
joblib.dump(emb_final, f'{MODEL_DIR}/emb_lr.joblib')
print(f'C = {EMB_C} | out-of-fold macro ROC-AUC: {best_emb[0]:.4f}')

#### Blend weights (chosen on out-of-fold predictions)

In [ ]:
grid = [(round(a, 2), round(b, 2), round(max(0.0, 1 - a - b), 2)) for a in np.arange(0, 1.0001, 0.05) for b in np.arange(0, 1.0001 - a, 0.05)]
blend = pd.DataFrame([{'finetuned': a, 'tfidf': b, 'embeddings': c,
                       'oof_macro_roc_auc': foldwise_score(Ydev, a * oof_ft + b * oof_tf + c * oof_emb, folds)} for a, b, c in grid])
blend = blend.sort_values('oof_macro_roc_auc', ascending=False).reset_index(drop=True)
W = blend.loc[0, ['finetuned', 'tfidf', 'embeddings']].astype(float).to_dict()
oof_final = W['finetuned'] * oof_ft + W['tfidf'] * oof_tf + W['embeddings'] * oof_emb
P_test = W['finetuned'] * P_test_ft + W['tfidf'] * P_test_tf + W['embeddings'] * P_test_emb
print('Blend weights:', W)
blend.head(5).round(4)

#### Thresholds (one per finding) and the test evaluation

In [ ]:
THR = finding_thresholds(Ydev, oof_final)
results = all_metrics(Ytest, P_test, THR)
print(f'Findings the note can flag: {(THR <= 1).sum()} of {len(CLASSES)}')
pd.Series(results).round(4).to_frame('test')

#### What each model adds (test macro ROC-AUC, 95% bootstrap CI)

In [ ]:
steps = {'Model 1: TF-IDF (train only)': P_test_ref,
         'Model 2: one fine-tuned model': test_ft[0],
         f'Model 2: ensemble of {N_ENSEMBLE}': P_test_ft,
         'Model 1: TF-IDF (train + val)': P_test_tf,
         'Model 3: embeddings + LR': P_test_emb,
         'Final blend': P_test}
comparison = pd.DataFrame([{'model': k, 'test_macro_roc_auc': main_score(Ytest, v), 'ci_95': '%.3f – %.3f' % tuple(boot_ci(Ytest, v)),
                            'test_micro_pr_auc': average_precision_score(Ytest.ravel(), v.ravel())} for k, v in steps.items()])
comparison.to_csv(f'{OUT_DIR}/model_comparison.csv', index=False)
comparison.round(4)

#### Per-finding report

In [ ]:
rows = [{'finding': c, 'test_cases': int(Ytest[:, j].sum()), 'roc_auc': roc_auc_score(Ytest[:, j], P_test[:, j]),
         'pr_auc': average_precision_score(Ytest[:, j], P_test[:, j]),
         'f1': f1_score(Ytest[:, j], (P_test[:, j] > THR[j]).astype(int), zero_division=0)}
        for j, c in enumerate(CLASSES) if MIN_POS <= Ytest[:, j].sum() < len(Ytest)]
per_finding = pd.DataFrame(rows).sort_values('test_cases', ascending=False).reset_index(drop=True)
per_finding.to_csv(f'{OUT_DIR}/per_finding_report.csv', index=False)
per_finding.round(3)

#### Charts: per-finding ROC-AUC and training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, max(4, 0.28 * len(per_finding))))
pf = per_finding.sort_values('roc_auc')
ax[0].barh(pf.finding, pf.roc_auc, color='#1428A0'); ax[0].axvline(0.5, c='k', ls='--', lw=.8)
ax[0].set_xlim(0.3, 1); ax[0].set_title('Test ROC-AUC per finding')
for k, h in enumerate(histories):
    h = pd.DataFrame(h); ax[1].plot(h.epoch, h.val_macro_roc_auc, 'o-', label=f'model {k + 1}')
ax[1].set_title('Held-out fold score per epoch'); ax[1].set_xlabel('epoch'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/final_evaluation.png', dpi=120); plt.show()

#### Log to MLflow

In [ ]:
with mlflow.start_run(run_name='final_model'):
    mlflow.log_params({**FINAL_HP, 'backbone': BEST_NAME, 'n_models': N_ENSEMBLE, 'emb_lr_C': EMB_C,
                       **{f'w_{k}': v for k, v in W.items()}, **{f'tfidf_{k}': v for k, v in TFIDF_CFG.items()}})
    mlflow.log_metrics({f'test_{k}': float(v) for k, v in results.items()})

### Leakage demonstration

The same model trained on the radiologist's `findings` text, which describes the X-ray (the answer).

In [ ]:
split_findings = lambda s: df[(df.split == s) & (df.findings_text.str.len() > 0)].reset_index(drop=True)
lf_train, lf_val, lf_test = split_findings('train'), split_findings('val'), split_findings('test')
leak_model, leak_tok, _, _ = train_model(BEST_NAME, {**FINAL_HP, 'epochs': min(4, MAX_EPOCHS)}, lf_train, lf_val, text_col='findings_text')
leak_score = main_score(Ymat(lf_test), predict_probs(leak_model, make_loader(lf_test, leak_tok, 64, False, 'findings_text')))
del leak_model; free()
with mlflow.start_run(run_name='leakage_demo_findings_input'):
    mlflow.log_metric('test_macro_roc_auc', leak_score)
print(f'Input = indication (honest): {results["macro_roc_auc"]:.3f}')
print(f'Input = findings   (leaky) : {leak_score:.3f}')

### Explanations

#### Save the model settings and load the model through the app API

In [ ]:
meta = {'backbone': BEST_NAME, 'classes': CLASSES, 'max_len': MAX_LEN, 'blend_weights': W,
        'thresholds': [float(t) for t in THR], 'hyperparameters': FINAL_HP, 'test_metrics': results}
json.dump(meta, open(f'{MODEL_DIR}/meta.json', 'w'), indent=2, default=float)
importlib.reload(pulmo_text)
tm = pulmo_text.TextModel(MODEL_DIR)

#### Highlighted words in example notes

In [ ]:
def show(text, label=None):
    ex, pr = tm.explain(text, label=label), tm.predict_text(text)
    color = lambda s: f'rgba(220,38,38,{min(abs(s), 1):.2f})' if s > 0 else f'rgba(37,99,235,{min(abs(s), 1):.2f})'
    words = ' '.join(f'<span style="background:{color(d["score"])};padding:1px 3px;border-radius:3px">{d["word"]}</span>' for d in ex['tokens'])
    top = ', '.join(f"{d['label']} {d['probability']:.2f}" for d in pr['top'][:3])
    display(HTML(f'<div style="font-family:sans-serif;margin:4px 0"><b>{ex["label"]}</b>: {words} <span style="color:#777">(top: {top})</span></div>'))

for t in test_df.text.iloc[np.argsort(-P_test.max(1))[:6]].tolist() + test_df.text.sample(3, random_state=SEED).tolist():
    show(t)

#### Words that push towards / away from "normal" (over the test notes)

In [ ]:
STOP = {'and', 'with', 'the', 'for', 'year', 'years', 'old', 'male', 'female', 'man', 'woman', 'patient', 'has', 'was',
        'are', 'not', 'from', 'this', 'that', 'his', 'her', 'hidden', 'had', 'have', 'been'}
REF = 'normal' if 'normal' in CLASSES else counts.index[0]
scores = defaultdict(list)
for t in test_df.text.head(60 if FAST else 300):
    for d in tm.explain(t, label=REF, n_steps=30)['tokens']:
        w = d['word'].lower()
        if w.isalpha() and len(w) > 2 and w not in STOP: scores[w].append(d['score'])
words = pd.DataFrame([(w, np.mean(v), len(v)) for w, v in scores.items() if len(v) >= 5], columns=['word', 'score', 'count'])
words.to_csv(f'{OUT_DIR}/word_attribution.csv', index=False)
top = pd.concat([words.nlargest(12, 'score'), words.nsmallest(12, 'score')]).drop_duplicates('word').sort_values('score')
plt.figure(figsize=(7, 7)); plt.barh(top.word, top.score, color=['#2563eb' if s > 0 else '#dc2626' for s in top.score])
plt.axvline(0, c='k', lw=.8); plt.title(f'Words pushing towards "{REF}" (blue) or away from it (red)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/word_attribution.png', dpi=120); plt.show()

### Outputs

#### Embeddings for every patient

In [ ]:
emb = tm.embed(df['text'].tolist())
emb_df = pd.DataFrame(emb, columns=[f'txt_{i}' for i in range(emb.shape[1])])
emb_df.insert(0, 'has_note', df.has_note.values); emb_df.insert(0, 'split', df.split.values); emb_df.insert(0, 'uid', df.uid.values)
emb_df.to_parquet(f'{OUT_DIR}/text_embeddings.parquet', index=False)

#### Predictions for every patient (out-of-fold for train/val, final model for test)

In [ ]:
pcols = [f'p_{c}' for c in CLASSES]
pred = pd.DataFrame(np.nan, index=range(len(df)), columns=pcols)
row = {u: i for i, u in enumerate(df.uid)}
pred.iloc[[row[u] for u in dev_df.uid]] = oof_final
pred.iloc[[row[u] for u in test_df.uid]] = P_test
pred.insert(0, 'has_note', df.has_note.values); pred.insert(0, 'split', df.split.values); pred.insert(0, 'uid', df.uid.values)
pred.to_csv(f'{OUT_DIR}/text_predictions.csv', index=False)
print('Embeddings', emb_df.shape, '| Predictions', pred.shape)

#### Zip everything

In [ ]:
if os.path.abspath('pulmo_text.py') != os.path.abspath(f'{OUT_DIR}/pulmo_text.py'):
    shutil.copy('pulmo_text.py', OUT_DIR)
with zipfile.ZipFile(f'{OUT_DIR}/text_model_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(MODEL_DIR):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT_DIR))
    for f in ['pulmo_text.py', 'text_embeddings.parquet', 'text_predictions.csv', 'model_comparison.csv', 'per_finding_report.csv', 'mlflow.db']:
        z.write(f'{OUT_DIR}/{f}', f)
    for f in glob.glob(f'{OUT_DIR}/figures/*.png'): z.write(f, os.path.relpath(f, OUT_DIR))
print('Saved text_model_outputs.zip')

### Final checks

#### The saved model reproduces the test score

In [ ]:
reloaded = main_score(Ytest, tm.predict_proba(test_df.text.tolist()))
print(f'Reloaded model: {reloaded:.4f} | in notebook: {results["macro_roc_auc"]:.4f}')

#### Example predictions

In [ ]:
for note in ['Routine pre-operative chest x-ray.', 'XXXX-year-old male with shortness of breath and productive cough, smoker.', '', None]:
    print(repr(note), '->', tm.predict_text(note))

#### Automatic checks

In [ ]:
assert abs(reloaded - results['macro_roc_auc']) < 0.01
assert results['micro_roc_auc'] > 0.5, 'probabilities are not calibrated across findings'
assert tm.predict_text(None)['has_note'] is False
assert len(tm.explain('cough and fever')['top']) > 0
assert emb_df.uid.is_unique and set(emb_df.split) == {'train', 'val', 'test'}
assert (emb_df.loc[~emb_df.has_note, emb_df.columns[3:]].abs().sum(axis=1) == 0).all()
assert pred.loc[pred.has_note, pcols].notna().all().all()
print('All checks passed.')